This code check the "androidmanifest.xml" for existance and at least one activity without cloning the repo

In [ ]:
import os
import re
import pandas as pd
import base64
import requests
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from time import sleep

# Load tokens from .env
load_dotenv("All_Tokens.env")
tokens = [v for k, v in os.environ.items() if k.startswith("GITHUB_TOKEN_") and v]
if not tokens:
    raise ValueError("No GitHub tokens found in All_Tokens.env")

token_index = 0
def get_headers():
    return {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-manifest-checker"
    }

def rotate_token():
    global token_index
    token_index = (token_index + 1) % len(tokens)
    print(f"🔁 Rotated to token #{token_index + 1}")

def extract_full_name_from_url(url):
    if isinstance(url, str) and "github.com" in url:
        return '/'.join(url.split("github.com/")[-1].split("/")[:2]).replace(".git", "")
    return None

# Input and output file paths
input_csv = r"D:\Android_Mobile_App\AndroidProject_dataset\Repo_List.csv"
output_csv = r"D:\Android_Mobile_App\AndroidProject_dataset\Repo_List_checked.csv"

# Load CSV and extract full_name
df = pd.read_csv(input_csv)
df["full_name"] = df["html_url"].apply(extract_full_name_from_url)
df["has_manifest"] = "no"
df["has_activity"] = "no"

# Only standard Android app manifest path
manifest_path = "app/src/main/AndroidManifest.xml"

# Main logic
for idx, row in df.iterrows():
    full_name = row.get("full_name")
    if not isinstance(full_name, str) or "/" not in full_name:
        print(f"⛔️ Skipping invalid full_name: {full_name}")
        continue

    print(f"\n🔍 Checking repo: {full_name}")

    url = f"https://api.github.com/repos/{full_name}/contents/{manifest_path}"
    print(f"📁 Trying path: {manifest_path}")

    success = False
    for _ in range(len(tokens)):
        response = requests.get(url, headers=get_headers())
        print(f"🔎 GET {url} → Status {response.status_code}")

        if response.status_code == 200:
            success = True
            break
        elif response.status_code == 403:
            print("⚠️ Rate limit hit, rotating token")
            rotate_token()
            sleep(1)
        elif response.status_code == 404:
            break
        else:
            print(f"❌ Unexpected error: {response.status_code}")
            break

    if not success or not response:
        print(f"❌ AndroidManifest.xml not found in {full_name}")
        continue

    df.at[idx, "has_manifest"] = "yes"

    try:
        content = response.json().get("content")
        if content:
            decoded = base64.b64decode(content).decode("utf-8")
            try:
                root = ET.fromstring(decoded)
                activities = root.findall(".//activity")
                if not activities:
                    ns_pattern = re.compile(r'\{(.+)\}')
                    match = ns_pattern.match(root.tag)
                    ns = {'android': match.group(1)} if match else {}
                    activities = root.findall(".//activity", namespaces=ns)
                if activities:
                    df.at[idx, "has_activity"] = "yes"
                    print(f"✅ Found activity tag in {full_name}")
                else:
                    print(f"⚠️ No <activity> found in manifest")
            except ET.ParseError:
                print("❌ XML parse error in manifest")
    except Exception as e:
        print(f"❌ Failed to decode content for {full_name}: {e}")
        continue

# Save output
df.to_csv(output_csv, index=False)
print(f"\n✅ Done. Results saved to: {output_csv}")


✅ Done. Results saved to: D:\Android_Mobile_App\AndroidProject_dataset\Repo_List_checked.csv
